In [1]:
import pandas as pd
import numpy as np
from nltk import sent_tokenize
from nltk import word_tokenize
from tqdm import tqdm
from Levenshtein import ratio
import os
import nltk
stop=nltk.corpus.stopwords.words('english')

In [27]:
df=pd.read_feather('/Volumes/T7/chroniclingamerica/kkk/revival_kkk_all.feather')

In [28]:
df.shape

(53614, 6)

In [10]:
df.head(2)

,city,state,date,lCCN,text,title
0,minneapolis,minnesota,1923-02-08,['2019271211'],VCa N uY i 0 Wiiv m voice KNIGHTS of the KU KL...,voice of the knights of the ku klux klan (minn...
1,aberdeen,mississippi,1921-01-28,['sn86074011'],jfllE AEECQEEIi WEEKLY V 77 TY tfYt W 7 I EVER...,"the aberdeen weekly (aberdeen, miss.) 1878-1933"


In [11]:
df['text'].iloc[2]

'IPWPJIIWP JvaajMiill II IIIHIRWWMPMMI ft V i Vf V t ki frl y iOUf mm MI Am Hi fflWMtSEMPEROR EULOGIZES DISORDER AftD feiil MAGISTRATES RAP Ink AAiniT Ar TUP ll ll rJflKIIUMIItMUA Ministers and Others Join in t Praiso of Evoning Publio Ledger for Expos lOTRY IS DENOUNCED Honest Courage of Expose Praised by Business Men To thu Editor ot Kutnlnp Public Ieiqrr Sir It nlwnyn was a known fnct thnt the mnrnltiR Pcmtc TKnonn utood for Juaticr tirnl rWitpmiMiem and It teem thnt Its ofNiirins the Evieino Pinuc Lepokr is fol lowing tlie footsteps of tlie fnther publication At n mcotlnc of the South Street Buitnesi Men Association held Thursday September 15 It wns Ilnnnlmoitslr revived thnt n vote of thanks be extended to your vnlun ble paper for It honest eonrnce and conviction In cxponltiK the Ku Klix Kln n ronglomerntlon of hoodlum and swindlers masquerading ns patriotic Amerlrans IIoplnB that you may carry on the food work of exposing the sorailed True Americans II M LEVY President S VRAM Vic

#### Sliding window of 3

In [4]:
candidate_article_index=[]
candidate_window_index=[]
candidate=[]
candidate_ratio=[]
for article_index, article in tqdm(df['text'].items(), total=len(df['text'])):
    tokens = [w for w in word_tokenize(str(article)) if w.strip()]  
    tokens = [t.lower() for t in tokens]
    windows = [
        ' '.join(tokens[i:i+3])   
        for i in range(len(tokens) - 2)
    ]
    for window in windows:
        ratio_value = ratio(window, 'ku klux klan')
        if (ratio_value >= 0.8) and (ratio_value < 1.0): #0.7 includes articles like "klux klan and" (0.72) or "klux klan in" (0.75)
            candidate_article_index.append(article_index)
            candidate_window_index.append(windows.index(window))
            candidate.append(window)
            candidate_ratio.append(ratio_value)

  0%|          | 0/53614 [00:00<?, ?it/s]

100%|██████████| 53614/53614 [00:24<00:00, 2194.33it/s]


In [17]:
candidate_df=pd.DataFrame({'article_index': candidate_article_index, 'window_index': candidate_window_index, 'window': candidate, 'ratio': candidate_ratio})

In [19]:
candidate_df[candidate_df['window']=='ku klux klnn']

,article_index,window_index,window,ratio
3,2,194,ku klux klnn,0.916667
226,142,165,ku klux klnn,0.916667
349,38,42,ku klux klnn,0.916667
432,22,180,ku klux klnn,0.916667
499,33,136,ku klux klnn,0.916667
521,28,125,ku klux klnn,0.916667
579,13,185,ku klux klnn,0.916667
585,70,167,ku klux klnn,0.916667
761,8,50,ku klux klnn,0.916667
786,144,45,ku klux klnn,0.916667


#### Find the context of the candidate three token

In [13]:
def extract_candidate_contexts(text, phrase, window):
    """
    Return list of context strings (joined tokens) for each match of phrase in text.
    Also returns the start token index for each match (useful for position).
    """
    if not phrase:
        return []

    tokens = word_tokenize(text.lower())
    phrase_tokens = word_tokenize(phrase.lower())
    windows = []
    n = len(tokens)
    m = len(phrase_tokens)

    # iterate only where phrase can fit
    for i in range(n - m + 1):
        if tokens[i:i + m] == phrase_tokens:
            start = max(0, i - window)
            end = min(n, i + m + window)   # use m (phrase length) not a fixed 3
            window_tokens = tokens[start:end]
            # you might want to show the phrase highlighted; here we just return the window text
            windows.append({
                'match_start_token_idx': i,
                'context_tokens': window_tokens,
                'context_text': " ".join(window_tokens)
            })
    return windows

In [29]:
results = []  # list of dicts

for article_index, article in tqdm(df[~df['text'].isna()]['text'].items(), total=len(df[~df['text'].isna()]['text'])):
    for candidate_phrase in pd.Series(candidate).unique():   # candidate assumed to be an iterable of phrases
        candidate_contexts = extract_candidate_contexts(article, candidate_phrase, window=5)
        if candidate_contexts:
            # append one result per found context to preserve alignment
            for ctx in candidate_contexts:
                results.append({
                    'article_index': article_index,
                    'candidate': candidate_phrase,
                    'match_start_token_idx': ctx['match_start_token_idx'],
                    'context_text': ctx['context_text'],
                    'context_tokens': ctx['context_tokens']
                })

100%|██████████| 52546/52546 [3:20:16<00:00,  4.37it/s]  


In [31]:
pd.DataFrame(results).to_feather('/Volumes/T7/chroniclingamerica/kkk/revival_kkk_contexts.feather')

In [26]:
df[~df['text'].isna()]

,city,state,date,lCCN,text,title
0,minneapolis,minnesota,1923-02-08,['2019271211'],VCa N uY i 0 Wiiv m voice KNIGHTS of the KU KL...,voice of the knights of the ku klux klan (minn...
1,aberdeen,mississippi,1921-01-28,['sn86074011'],jfllE AEECQEEIi WEEKLY V 77 TY tfYt W 7 I EVER...,"the aberdeen weekly (aberdeen, miss.) 1878-1933"
2,philadelphia,pennsylvania,1921-09-20,['sn83045211'],IPWPJIIWP JvaajMiill II IIIHIRWWMPMMI ft V i V...,evening public ledger (philadelphia [pa.]) 191...
3,indianapolis,indiana,1924-10-30,['sn82015313'],THLKfcSJJAY OCT 301024 KLAN GIVES SLATE IN CON...,the indianapolis times (indianapolis [ind.]) 1...
4,philadelphia,pennsylvania,1921-09-17,['sn83045211'],r i i j lv 3T i t EVENING PUBLIC LEDERPfilLADE...,evening public ledger (philadelphia [pa.]) 191...
...,...,...,...,...,...,...
155,indianapolis,indiana,1925-11-13,['sn82015313'],Home Edition TIE Times weekly American Legion ...,the indianapolis times (indianapolis [ind.]) 1...
156,indianapolis,indiana,1928-11-01,['sn82015313'],Second Section 15 WILL FACE COURT IN AUTO THEF...,the indianapolis times (indianapolis [ind.]) 1...
157,west union,ohio,1923-09-13,['sn83035189'],THE PEOPLES DEFENDER VOLUME LVIII WEST UNION O...,"the people's defender (west union, adams count..."
158,indianapolis,indiana,1926-10-11,['sn82015313'],Home Edition You Have to Iload The Times If Yo...,the indianapolis times (indianapolis [ind.]) 1...


In [61]:
candidate_context_list

['m voice knights of the ku klux kl an volume i believing that',
 'voice knights of the ku klux kl an volume i believing that all',
 'm voice knights of the ku klux kl an volume i believing that']

In [22]:
candidate_tokens=[]
for three_token in candidate:
    # print(three_token)
    for token in word_tokenize(three_token):
        # print(token)
        if token not in stop:
            candidate_tokens.append(token)

In [28]:
len(pd.Series(candidate_tokens).unique())

417

In [36]:
pd.Series(candidate_tokens).value_counts().keys()

Index(['klan', 'klux', 'ku', 'kuklux', 'klans', 'kiux', 'klu', 'kian',
       'klansmen', 'klnn',
       ...
       'inu', 'kiuz', 'proku', 'long', 'klnz', 'klaz', 'nj', 'fcian', 'hilan',
       'mob'],
      dtype='object', length=417)

#### Find -5 +5 context of the candidate token

In [45]:
article_index_list=[]
candidate_context_list=[]
for candidate_token in tqdm(pd.Series(candidate_tokens).value_counts().keys()):
    for article_index, article in df['text'].items():
        tokens = [w for w in word_tokenize(str(article)) if w.strip()] #article token list
        tokens = [t.lower() for t in tokens] #lowercase article token list
        if candidate_token in tokens:
            #print token index
            token_indices = [i for i, x in enumerate(tokens) if x == candidate_token]
            for token_index in token_indices:
                context_window = tokens[max(0, token_index-5):min(len(tokens), token_index+6)]
                candidate_context_list.append(' '.join(context_window))
                article_index_list.append(article_index)

 13%|█▎        | 56/417 [11:24<1:13:34, 12.23s/it]


KeyboardInterrupt: 

In [42]:
from collections import defaultdict
from nltk.tokenize import word_tokenize
from tqdm import tqdm

def find_sublist_indices(tokens, pattern):
    """Return start indices where pattern (a list) appears in tokens."""
    m = len(pattern)
    if m == 0: return []
    return [i for i in range(len(tokens)-m+1) if tokens[i:i+m] == pattern]

def merge_spans(spans):
    """Merge overlapping/adjacent (start, end) spans (end exclusive)."""
    if not spans: return []
    spans = sorted(spans)
    merged = [list(spans[0])]
    for s,e in spans[1:]:
        if s <= merged[-1][1]:             # overlap/adjacent
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s,e])
    return [(a,b) for a,b in merged]

def get_consolidated_contexts(df_text_series, candidate_token,
                              before=10, after=10, lowercase=True):
    """
    Return dict: article_index -> list of consolidated context strings.
    candidate_token may be multi-word (e.g. "ku klux klan").
    """
    cand_tokens = word_tokenize(candidate_token)
    if lowercase:
        cand_tokens = [t.lower() for t in cand_tokens]

    results = defaultdict(list)

    for idx, article in df_text_series.items():
        tokens = [t for t in word_tokenize(str(article)) if t.strip()]
        tokens_lc = [t.lower() for t in tokens] if lowercase else tokens

        starts = find_sublist_indices(tokens_lc, cand_tokens)
        if not starts:
            continue

        spans = []
        m = len(cand_tokens)
        n = len(tokens)
        for st in starts:
            s = max(0, st - before)
            e = min(n, st + m + after)
            spans.append((s,e))

        for s,e in merge_spans(spans):
            text = " ".join(tokens[s:e])
            results[idx].append(f"[start] {text} [end]")

    return dict(results)

# --- Example usage (integrate into your loop) ---
no_overlap_context = []
for candidate_token in tqdm(pd.Series(candidate_tokens).value_counts().keys()):
    consolidated = get_consolidated_contexts(df['text'], candidate_token,
                                             before=10, after=10)
    for article_index, contexts in consolidated.items():
        for ctxt in contexts:
            no_overlap_context.append({
                "candidate": candidate_token,
                "article_index": article_index,
                "context": ctxt
            })


  0%|          | 2/417 [00:25<1:29:53, 13.00s/it]


KeyboardInterrupt: 

In [43]:
no_overlap_context

[{'candidate': 'klan',
  'article_index': 0,
  'context': '[start] Activities and Ideals of the Knights of the Ku Klux Klan North Star Klan No 2 of Minneapolis have caused this sheet to be issued A DEFENCE OF THE KU KLUX KLAN Reprint from Literary Digest Jan 20 1923 The Ku Klux Klan has been charged with all sorts of crimes and mis [end]'},
 {'candidate': 'klan',
  'article_index': 0,
  'context': '[start] of The Digest for in stance Louisiana members of the Klan were charged by Louisiana editors with the murder of two [end]'},
 {'candidate': 'klan',
  'article_index': 0,
  'context': '[start] this horrible crime had been committed by members of the Klan there was no specific defense of the murderers in particular or the Klan in general to be found in either Northern or South ern newspapers We heard on the other hand that the Klan had usurped the [end]'},
 {'candidate': 'klan',
  'article_index': 0,
  'context': '[start] from an automobile on aprominent street WASHINGTON The Ku Klux Kl

In [ ]:
klan_ocr_error=[
    'kuklux', 'klans', 'kiux', 'klu', 'kian', 'klansmen', 'klnn', 'antiku', 'kn', 'kan',
    'klnx', 'klun', 'klau', 'ka', 'kux', 'kit', 'klax', 'klin', 'ki', 'khix', 
    'kii', 'ko', 'kl', 'kluxklan', 'klanism', 'klanu', 'kltix', 'kun', 'klansman', 'xu',
    'kln', 'klus', 'elan', 'kluk', 'kti', 'lux', 'klui', 'klaa', 'klar', 'kjux', 
    'klim', 'ivlan', 'klttx', 'kin', 'kjan', 'klah', 'hi', 'hu', 'ktux', 'rian', 
    'xlux', 'kluz', 'ivlux', 'clan', 'klix', 'iclan', 'klaag', 'kilux', 'kli', 'iklux',
    'antlku', 'klur', 'klox', 'klsn', 'blackku', 'xian', 'khan', 'klxn', 'kluklux', 'mux', 
    'kinn', 'kians', 'klni', 'kliix', 'kiu', 'kinx', 'klandidate', 'ikiux', 'kltn', 'kldx', 
    'rlux', 'klari', 'klaners', 'kant' ,'kluv', 'klam', 'rlan', 'klak' ,'klanem', 'kliu', 
    'klanhave', 'kla', 'cu', 'klanex', 'klop', 'kilan', 'xlan', 'iku', 'iflux', 'klon', 
    'su', 'iclux',' kill', 'kluan', 'khun', 'klati', 'khin', 'km', 'exku', 'klqx', 
    'kliijx', 'klilx', 'kfua', 'itlan', 'klart', 'ktan', 'klnu', 'elux', 'kluti', 'klmi', 
    'kll', 'hlux', 'kleaning', 'kllux', 'klanthe', 'ilian', 'klanstate', 'kain', 'kino', 
    







]

klan        935
klux        828
ku          820
kuklux      415
klans        58
kiux         56
klu          53
kian         51
klansmen     35
klnn         31
antiku       27
kn           26
kan          25
klnx         18
k            18
klun         16
klau         15
u            14
ka           12
kux          11
kit          11
klax         11
klin         11
ki           10
khix         10
kii           9
ko            9
kl            9
kluxklan      8
klanism       8
klanu         8
kltix         8
kun           7
klansman      7
xu            7
man           7
kln           7
klus          7
elan          7
kluk          6
lan           6
kti           6
lux           6
klui          6
klaa          6
klar          6
warns         5
kjux          5
old           5
klim          5
j             5
ivlan         4
klttx         4
kin           4
kjan          4
Name: count, dtype: int64

In [56]:
candidate_article_index = []
candidate_window_index = []     # window index (same as token index)
candidate = []
candidate_index = []            # index of FIRST sliding-window start per article

for article_index, article in tqdm(df['text'].items(), total=len(df)):
    tokens = [w for w in word_tokenize(str(article)) if w.strip()]  

    first_token_idx = None  # store token index of first match

    for i in range(len(tokens) - 2):     # i = index of first token in window
        window = ' '.join(tokens[i:i+3])

        if ratio(window.lower(), 'ku klux klan') >= 0.8:

            candidate_article_index.append(article_index)
            candidate_window_index.append(i)   # the window starts at token i
            candidate.append(window)

            if first_token_idx is None:
                first_token_idx = i

    candidate_index.append(first_token_idx)

100%|██████████| 53614/53614 [00:24<00:00, 2196.54it/s]


In [57]:
pd.DataFrame({'article_index': candidate_article_index, 'window_index': candidate_window_index, 'window': candidate, 'first_token_index': candidate_index})

ValueError: All arrays must be of the same length

In [46]:
df.iloc[84]['text']

'TfIECAIULQgTHBNMaB V Pop Says If Sopers Cant Get Law Violators in St Paul the ku klux klan LAN CANDIDATE IS ELECTED MAYOR Fort Smith Kan A letter of con gratulation bearing the signatures of all the county and city officials of Haskell county was forwarded to Mayorelect David L Ford alleged Ku Klux Klan candidate who was elected mayor of Fort Smith by one of the largest majorities ever given a candidate The vote was 1824 to 524 A bitter fight had been waged against Ford and a day before the election large page circulars attack ing Ford and the Klan were distrib uted all over Fort Smith THREEMILE KLAN PARADE IN ILLINOIS Benton 111The citizens of John ston City West Frankfort and Ben ton were thrown into a quiver of ex citement by the invasion of a mon ster parade of Knights of the In visible Empire It was a demonstra tion to partially show the strength of the organization in this section Those who saw the parade declare it was three miles long and that nearly a thousand cars were in li

In [52]:
for index, item in enumerate(candidate_article_index):
    if item == 84:
        print(index)

0
74
149


In [55]:
candidate[149]

'ku klux klan'

In [ ]:
def keyword_search(df: pd.DataFrame, substring: str):
    state = []
    city = []
    date = []
    lccn = []
    sent_list = []
    title = []
    context_list = []
    text = []

    for idx, val in df.iterrows():
        tokenized_sent = word_tokenize(val['text'].lower())
        matching_items = [item for item in tokenized_sent if substring in item]
        to_context = []
        for num, sent in enumerate(tokenized_sent):
            interim_to_context = []
            if substring in sent:
                interim_to_context.extend(tokenized_sent[max(0, num - 20): num + 20])
                to_context.append(interim_to_context)

        state.append(val['state'])
        city.append(val['city'])
        date.append(val['date'])
        lccn.append(val['lCCN'])
        title.append(val['title'])
        sent_list.append(matching_items)
        context_list.append(to_context)
        text.append(val['text'])

    newdf = pd.DataFrame({'state': state, 'city': city, 'date': date, 'lccn': lccn, 'sent': sent_list, 'title': title, 'context': context_list, 'text': text})
    return newdf

In [ ]:
[item for item in word_tokenize(df['text'].iloc[0].lower()) if 'klan' in item]

In [ ]:
df['text'].iloc[100]

In [ ]:
path='/Volumes/T7/chroniclingamerica/kkk/revival/'
append_df=[]
for i in tqdm(os.listdir(path)):
    if i.startswith('._'):
        continue
    elif i.endswith('.csv'):
        df=pd.read_csv(path+i)
        df['text']=df['text'].astype(str)
        newdf=keyword_search(df, 'klan')
        lendf=newdf[newdf['sent'].apply(lambda x: len(x) > 0)]
        append_df.append(lendf)
        newdf=keyword_search(df, 'ku')
        lendf=newdf[newdf['sent'].apply(lambda x: len(x) > 0)]
        append_df.append(lendf)
        newdf=keyword_search(df, 'klux')
        lendf=newdf[newdf['sent'].apply(lambda x: len(x) > 0)]
        append_df.append(lendf)

In [ ]:
kkkdf=pd.concat(append_df).reset_index(drop=True)
kkkdf=kkkdf.explode(['sent', 'context'], ignore_index=True)

In [ ]:
kkkdf

In [ ]:
kkkdf.to_feather('/Volumes/T7/chroniclingamerica/kkk/revival_kkk_context')